# Day 025：GRPO rollout、reward 与组内相对优势

本 Notebook 对照 `train_grpo.py` 和 `rollout_engine.py`，只保留 GRPO 总体链路与小型数值实验。
今天的掌握边界是正确区分 policy、reference、reward model、old policy 和 group advantage。

## 1. GRPO 的核心链路

`prompt -> policy 生成多条回答 -> reward model/规则评分 -> 组内比较 -> advantage -> 更新 policy`。
数据集仍然提供 prompt，但不需要提前提供 chosen/rejected 回答。reference 不生成另一组回答，
只提供 KL 基线；old policy 概率记录回答生成时的行为，用于 ratio。

In [ ]:
import torch

# 两个 prompt，每个 prompt 有三个 rollout 回答
rewards = torch.tensor([1.2, 0.4, 0.8, 0.1, -0.2, 0.5])
num_generations = 3
grouped = rewards.view(-1, num_generations)
mean = grouped.mean(dim=1).repeat_interleave(num_generations)
std = grouped.std(dim=1, unbiased=False).repeat_interleave(num_generations)
advantages = (rewards - mean) / (std + 1e-4)
print('grouped rewards:', grouped)
print('advantages:', advantages)

组内 advantage 为正的回答高于同 prompt 的平均水平，倾向提高概率；为负的回答低于平均水平，倾向降低概率。
`1e-4` 防止整组 reward 相同时除零。

In [ ]:
# ratio 与 KL 的最小数值示意
old_logp = torch.tensor([-2.0, -1.0])
current_logp = torch.tensor([-1.5, -1.2])
reference_logp = torch.tensor([-1.8, -1.1])
ratio = torch.exp(current_logp - old_logp)
kl_estimate = torch.exp(reference_logp - current_logp) - (reference_logp - current_logp) - 1
print('ratio:', ratio)
print('per-token KL estimate:', kl_estimate)

`ratio = current/old` 衡量 policy 相对生成行为时改变了多少；KL 项约束当前 policy 不要偏离 reference 过远。
今天没有伪造真实 GRPO 训练输出，完整源码说明和验收边界见 Day 025 Markdown。
下一恢复点：`train_ppo.py` 的 actor、reference、reward、critic 初始化和 PPO 训练入口。